# Convex Analysis & Lagrange Duality : Practice & Experiments

**Authored by Alexandre Mathias DONNAT, Sr - Télécom Paris**


The goal of this session is to become familiar with the concept of Lagrangian duality by relying on the support vector machines problem.

Given observations $(x_i, y_i)_{1 \leq i \leq N}$, it is written as:

$$\min_{w \in \mathbb{R}^d} C \sum_{i=1}^{N} \max(0, 1 - y_i x_i^T w) + \frac{1}{2} \|w\|_2^2 \tag{1}$$

This function is convex but introduces the maximum, which is not differentiable. We can write it equivalently with constraints:

$$\min_{w \in \mathbb{R}^d, \xi \in \mathbb{R}^N} C \sum_{i=1}^{N} \xi_i + \frac{1}{2} \|w\|_2^2 \tag{2}$$

subject to:
- $\xi_i \geq 1 - y_i x_i^T w, \quad \forall i \in \{1, \ldots, N\}$
- $\xi_i \geq 0, \quad \forall i \in \{1, \ldots, N\}$

We introduce the Lagrangian defined for $w \in \mathbb{R}^d$, $\xi \geq 0$ and $\phi \in \mathbb{R}^N$, $\phi \geq 0$ by:

$$L(w, \xi, \phi) = C \sum_{i=1}^{N} \xi_i + \frac{1}{2} \|w\|_2^2 + \sum_{i=1}^{N} \phi_i(1 - y_i x_i^T w - \xi_i)$$

By maximizing $L$ with respect to $w$ and $\xi$, we find the dual function defined for $\phi \in [0, C]^N$ by:

$$D(\phi) = -\frac{1}{2} \left\| \sum_{i=1}^{N} \phi_i y_i x_i \right\|^2 + \sum_{i=1}^{N} \phi_i \tag{3}$$

As in the second part of the SGD practical session, we will use the Iris dataset for numerical experiments.


## Questions

1. We will first work on formulation (1). Although the function is not differentiable, it is differentiable almost everywhere. Show that for any $w$ such that $1 - y_i x_i^\top w \ne 0$, the gradient of the function $F_i(w) = \max(0, 1 - y_i x_i^\top w)$ is given by  
$$
\nabla F_i(w) =
\begin{cases}
0 & \text{if } 1 - y_i x_i^\top w < 0, \\
- y_i x_i & \text{if } 1 - y_i x_i^\top w > 0.
\end{cases}
$$


Let $m_i(w) = 1 - y_i x_i^\top w$.

Then $F_i(w) = \max(0, m_i(w))$.

**Case 1:** $m_i(w) < 0$  
Then $F_i(w) = 0$ locally, so the gradient is zero.

**Case 2:** $m_i(w) > 0$  
Then $F_i(w) = m_i(w) = 1 - y_i x_i^\top w$.  
Thus the gradient is $-y_i x_i$.

**Case 3:** $m_i(w) = 0$  
The function is not differentiable at this point (kink of the max).  
But the statement excludes this case.


2. Review the SGD practical session, especially question 12. Implement the stochastic gradient method for problem (1). Do not worry about the non-differentiability of the function for the implementation.




Problem (1) is:

$$f(w) = C \sum_{i=1}^{N} \max(0, 1 - y_i x_i^T w) + \frac{1}{2} \|w\|_2^2$$

A stochastic gradient method consists of:

1) Draw a random index $i$
2) Compute the partial gradient
3) Update $w$

Stochastic gradient:

$$g(w; i) = w + C \nabla F_i(w)$$

Update rule:

$$w \leftarrow w - \eta_k g(w; i)$$

with a decreasing step size, for example:

$$\eta_k = \frac{\eta_0}{\sqrt{k+1}}$$

In [14]:
import numpy as np
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = iris.data
y_raw = iris.target

mask = (y_raw < 2)
X = X[mask]
y_raw = y_raw[mask]

X = StandardScaler().fit_transform(X)

y = np.where(y_raw == 0, -1.0, 1.0)

N, d = X.shape
C = 1.0

def primal_value(w, X, y, C):
    margins = 1 - y * (X @ w)
    return C * np.maximum(0, margins).sum() + 0.5 * np.dot(w, w)

def sgd_primal(X, y, C, n_iters=30000, eta0=0.5, seed=0):
    rng = np.random.default_rng(seed)
    N, d = X.shape
    w = np.zeros(d)

    for k in range(n_iters):
        i = rng.integers(N)
        margin = 1 - y[i] * (X[i] @ w)

        if margin > 0:
            grad_Fi = -y[i] * X[i]
        else:
            grad_Fi = np.zeros(d)

        g = w + C * grad_Fi
        eta = eta0 / np.sqrt(k + 1)
        w -= eta * g

    return w

3. Write the dual problem in the form

$$\max_{\phi \in [0, C]^N} q(\phi)$$

where $q$ is a quadratic function whose gradient we will compute.


We define the matrix $Q$ by:

$$Q_{ij} = y_i y_j x_i^T x_j$$

The dual problem becomes:

$$q(\phi) = -\frac{1}{2} \phi^T Q \phi + \mathbf{1}^T \phi$$

with the constraint:

$$0 \leq \phi_i \leq C \quad \forall i$$

The gradient is:

$$\nabla q(\phi) = -Q \phi + \mathbf{1}$$

In [15]:
def build_Q(X, y):
    return (y[:, None] * y[None, :]) * (X @ X.T)

def q(phi, Q):
    return -0.5 * phi @ (Q @ phi) + phi.sum()

def grad_q(phi, Q):
    return -Q @ phi + np.ones_like(phi)

4. Implement projected gradient ascent:

$$
\phi^{k+1} = \Pi_{[0, C]^N}\bigl(\phi^k + \gamma \nabla q(\phi^k)\bigr)
$$

Choose a step size $\gamma$ that guarantees convergence.

The gradient of $q$ is Lipschitz with constant  
$$
L = \|Q\|_2 = \lambda_{\max}(Q).
$$

A standard convergence condition is  
$$
0 < \gamma \le \frac{1}{L}.
$$

We therefore choose  
$$
\gamma = \frac{0.99}{\lambda_{\max}(Q)}.
$$

The projection onto $[0, C]^N$ is done coordinate-wise by clipping each entry between $0$ and $C$.

In [16]:
def proj_box(phi, C):
    return np.clip(phi, 0, C)

def projected_grad_ascent(Q, C, n_iters=10000):
    N = Q.shape[0]
    phi = np.random.rand(N) * C

    L = np.linalg.eigvalsh(Q).max()
    gamma = 0.99 / (L + 1e-12)

    for _ in range(n_iters):
        phi = proj_box(phi + gamma * grad_q(phi, Q), C)

    return phi

Q = build_Q(X, y)

phi_star = projected_grad_ascent(Q, C, n_iters=10000)

5. The KKT conditions give:

$$w^* = \sum_{i=1}^{N} \phi_i^* y_i x_i$$

Verify if the primal and dual solutions are compatible.
Compare the primal and dual objective values.
Connect this to weak duality.


We reconstruct $w_{\text{KKT}}$:

$$w_{\text{KKT}} = \sum_{i} \phi_i y_i x_i$$

We compare:

- $w_{\text{SGD}}$ (primal solution)
- $w_{\text{KKT}}$ (reconstructed dual solution)

If the algorithms have converged, these two vectors should be close.

**Weak duality:**

For any feasible dual solution $\phi$:

$$D(\phi) \leq P(w^*)$$

Numerically, we should observe:

$$q(\phi^*) \leq f(w_{\text{SGD}})$$

If this is not the case, there is a convergence or implementation issue.


In [ ]:

# Primal via SGD
w_sgd = sgd_primal(X, y, C, n_iters=30000, eta0=0.5, seed=0)

# Dual via projected gradient ascent
Q = build_Q(X, y)
phi_star = projected_grad_ascent(Q, C, n_iters=10000)

# Reconstruct w from dual (KKT)
w_kkt = w_from_phi(phi_star, X, y)

# Compare primal/dual values
print("||w_sgd - w_kkt|| =", np.linalg.norm(w_sgd - w_kkt))
print("Primal(w_sgd) =", primal_value(w_sgd, X, y, C))
print("Primal(w_kkt) =", primal_value(w_kkt, X, y, C))
print("Dual(phi*)   =", q(phi_star, Q))

||w_sgd - w_kkt|| = 0.6036746739524029
Primal(w_sgd) = 6.531450210302467
Primal(w_kkt) = 0.7061459078358527
Dual(phi*)   = 0.7054194533663003


6. What do you notice that is particular about the dual solution?

The dual solution φ* exhibits two characteristic properties of SVMs:

It is generally sparse:

- Many coefficients $\phi_i^*$ are exactly zero.
- Only a small number of $\phi_i^*$ are strictly positive.
- These indices correspond to the **support vectors**: only they contribute to $w^* = \sum_i \phi_i^* y_i x_i$.

Many coefficients lie on the boundaries (0 or C):

- $\phi_i^* = 0$: the point does not influence the solution (well classified, far from the margin),
- $0 < \phi_i^* < C$: the point lies exactly on the margin,
- $\phi_i^* = C$: the point violates the margin constraint (inside the margin or misclassified in the soft-margin case).

In our numerical results, we also observe that the dual solution is highly consistent:

$$\text{Dual}(\phi^*) \approx \text{Primal}(w_{\text{KKT}})$$

which indicates a very small duality gap and therefore a nearly optimal dual solution.

In contrast, the SGD primal solution gives a much larger objective value, suggesting that the stochastic gradient method has not fully converged.
